<a href="https://colab.research.google.com/github/nicolastibata/MINE_4210_ADL_202620/blob/main/assingments/assingment_3/MINE_4210_ADL_202620_T3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Logo ADL](https://github.com/nicolastibata/MINE_4210_ADL_202620/blob/main/docs/logo.png?raw=true)


# **Taller 3: Clasificación de textos con modelos encoder**

### **Contexto y Objetivos**

En 2015, la Organización de las Naciones Unidas (ONU) adoptó la Agenda 2030, que establece 17 Objetivos de Desarrollo Sostenible (ODS) y 169 metas orientadas a enfrentar desafíos sociales, económicos y ambientales. En este contexto, organizaciones como el Fondo de Población de las Naciones Unidas (UNFPA) trabajan con instituciones públicas para identificar problemas y evaluar políticas mediante mecanismos de participación ciudadana. Uno de los principales desafíos es analizar la información textual recopilada en estos procesos, ya que requiere tiempo, recursos y conocimiento experto para relacionarla con los ODS. Automatizar este análisis facilitaría la identificación de las necesidades de la población y apoyaría la toma de decisiones y la formulación de políticas públicas alineadas con la Agenda 2030.

En el Laboratorio 6 este problema se resolvió con una red LSTM y una capa de embeddings aprendida desde cero. En este taller utilizarán BETO, un modelo BERT preentrenado en español, para analizar cómo cambia el desempeño cuando el modelo parte de representaciones contextuales preentrenadas. También compararán dos formas de aprovechar el modelo: como extractor de características fijo y mediante ajuste fino (fine-tuning).

<br>

**Conjunto de datos:** Se utilizará el archivo TextosODS.xlsx (los del [lab6](https://github.com/nicolastibata/MINE_4210_ADL_202620/blob/main/labs/Laboratorio_6/TextosODS.xlsx)), que contiene 9.656 fragmentos de textos (columna textos) etiquetados con uno de los ODS 1 a 16 (columna ODS). Para el entrenamiento, conviertan las etiquetas al rango 0–15. Este conjunto permite estudiar la clasificación temática por ODS, aunque sus textos académicos no representan necesariamente los textos obtenidos en procesos de participación ciudadana.

Dividan los datos en entrenamiento (70 %), validación (15 %) y prueba (15 %) de forma estratificada, con random_state=42. Pueden separar primero el 15 % para prueba y después reservar 15/85 de los datos restantes para validación. Utilicen las mismas particiones en todos los experimentos.

<br>

**Actividades**

**1. Exploración previa:** Antes de entrenar los modelos.

- Calculen la distribución de la longitud de los textos en tokens de BETO, incluyendo los tokens especiales y antes del truncamiento, y el porcentaje que supera 256 tokens.
- Revisen la distribución de clases.
- Construyan un diagrama de cajas de la longitud de los textos (en tokens) por ODS.

<br>

**2. Modelo para construir:** El objetivo es construir y comparar los siguientes enfoques.

A diferencia del modelo utilizado en el Laboratorio de BERT, que está en inglés y se carga desde TF Hub, BETO está disponible en Hugging Face (dccuchile/bert-base-spanish-wwm-cased). Para los modelos 2 y 3 utilizarán la librería transformers con PyTorch. El siguiente ejemplo muestra únicamente cómo cargar el tokenizador y el modelo base; deberán investigar cómo extraer representaciones y cómo realizar el ajuste fino. Indiquen en el notebook las fuentes que consultaron.

```python
from transformers import AutoTokenizer, AutoModel
MODELO = "dccuchile/bert-base-spanish-wwm-cased"
# Cargar el tokenizador y el modelo
tokenizer = AutoTokenizer.from_pretrained(MODELO)
model = AutoModel.from_pretrained(MODELO)
# Ejemplo de tokenización
texto = "La educación de calidad es fundamental para el desarrollo sostenible."
entradas = tokenizer(texto, return_tensors="pt")
print(tokenizer.convert_ids_to_tokens(entradas["input_ids"][0]))
```

- Modelo 1 (línea base): la LSTM del Laboratorio 6. Pueden reutilizar su código, pero deben reentrenarla con las particiones de este taller y reportar las mismas métricas que para los demás modelos.

- Modelo 2 (BETO congelado):

  - Modelo: dccuchile/bert-base-spanish-wwm-cased, sin modificar sus pesos.

  - Representación de cada texto: promedio de los vectores de la última capa, ponderado por la máscara de atención (mean pooling), lo que produce un vector de 768 dimensiones.

  - Cabeza de clasificación: Dense(256, activation='relu') + Dropout(0.3) + Dense(16, activation='softmax').

  - Entrenamiento: máximo 20 épocas, con EarlyStopping sobre val_loss y patience=4.

  - **Sugerencia:** calculen las representaciones una sola vez y guárdenlas, de modo que el entrenamiento de la cabeza no requiera volver a pasar los textos por BETO.

- Modelo 3 (BETO con ajuste fino):

  - AutoModelForSequenceClassification con el mismo modelo base y 16 clases.

  - Congelen los embeddings y los primeros 10 bloques encoder. Entrenen únicamente los últimos 2 bloques encoder y la cabeza de clasificación.

La siguiente tabla resume las diferencias entre los tres enfoques:

| Modelo | Representación del texto | Pesos del encoder | Cabeza de clasificación |
|-|-|-|-|
| Modelo 1 | Embeddings aprendidos desde cero + LSTM | Se entrenan desde cero | La del Laboratorio 6 |
| Modelo 2 | BETO (mean pooling) | Congelados | MLP |
| Modelo 3 | BETO | Solo los últimos 2 bloques encoder (fine-tuning) | Capa lineal de AutoModelForSequenceClassification |

**Configuración:**
- max_length = 256 para los modelos 2 y 3.
- Modelo 3 (fine tuning parcial). Tasa de aprendizaje 2e-5, tamaño de lote 16, máximo 3 épocas y weight decay 0,01. Pueden utilizar precisión mixta (FP16) para reducir el consumo de memoria y acelerar el entrenamiento si la GPU lo permite. Conserven el checkpoint con mayor F1 macro en validación y utilícenlo para evaluar el conjunto de prueba.
- Utilicen GPU (por ejemplo, Colab con T4).

<br>

**3. Resultados**. Para cada modelo reporten:

- Accuracy y F1 macro en el conjunto de prueba.
- F1-score por clase.
- Matriz de confusión.
- Número de parámetros entrenables y tiempo aproximado de entrenamiento. Para el Modelo 2, reporten por separado el tiempo de extracción de embeddings y el de entrenamiento de la cabeza de clasificación.
- Curvas de loss de entrenamiento y validación.
Presenten los resultados en una tabla que permita comparar los tres enfoques.

<br>

**4. Análisis**. A partir de los resultados, respondan:

1. ¿Cómo se ordenan los tres modelos según su desempeño? ¿Qué diferencias en la forma de representar el texto explican ese orden? ¿En qué se parece la comparación entre la LSTM y BETO congelado a la que hicieron en el Taller 2 entre la CNN desde cero y transfer learning?
2. ¿Qué diferencias encuentran entre el desempeño de BETO congelado y BETO con ajuste fino? ¿Cómo influye la adaptación de las representaciones a la tarea de clasificación?
3. Identifiquen los pares de ODS que más se confunden. Revisen 5 errores del mejor modelo y clasifíquenlos en: error del modelo, ambigüedad temática o etiqueta potencialmente dudosa (sin asumir que está incorrecta). ¿Qué indican estos casos sobre formular el problema con una sola etiqueta por texto?
4. Según el diagrama de cajas, ¿hay ODS cuyos textos sean claramente más largos o cortos que los demás? ¿Qué implicaría esto si un modelo aprendiera a usar la longitud como señal para clasificar?
5. Si una organización necesitara clasificar miles de documentos con recursos de cómputo limitados, ¿qué enfoque recomendarían y por qué? Distingan el costo de entrenamiento del costo de inferencia.

**Bono (hasta 15 % adicional).** Repitan el ajuste fino parcial del Modelo 3 con MarIA (BSC-LT/roberta-base-bne, congelando los embeddings y los primeros 10 bloques encoder. Reporten las mismas métricas y respondan: ¿mejora el desempeño frente a BETO? ¿Qué diferencias entre los dos modelos (corpus de preentrenamiento, tokenizador, procedimiento de preentrenamiento) podrían explicar el resultado?


### **Entregable**

Notebook (.ipynb) con todas las celdas ejecutadas y con todas las indicaciones anteriormente estipuladas en la sección de laboratorios en **Bloque Neón**.



### **Rúbrica**

| Criterio | Peso | Qué se evalúa |
|-|-|-|
| Exploración previa | 10 % | Distribución de longitudes en tokens, distribución de clases y análisis de longitud por ODS. |
| Modelo 1. LSTM | 10 % | Reentrenamiento correcto con las particiones del taller. |
| Modelo 2. BETO congelado | 20 % | Extracción correcta de representaciones (mean pooling con máscara de atención) y entrenamiento de la cabeza MLP. |
| Modelo 3. BETO con ajuste fino | 20 % | Carga y configuración correcta de BETO para clasificación, entrenamiento completo y selección del modelo con el conjunto de validación. |
| Evaluación y presentación de resultados | 10 % | Métricas, matrices de confusión y costo computacional, organizados para facilitar la comparación. |
| Análisis e interpretación | 30 % | Comparación justificada entre enfoques, análisis de errores y discusión sobre atajos y sobre la formulación del problema. |
| Bono. MarIA | +15 % | Entrenamiento con la misma configuración, reporte de métricas y comparación justificada con BETO. |